<img src="https://raw.githubusercontent.com/IDEALLab/EngiOpt/codex/dcc26-workshop-notebooks/workshops/dcc26/assets/engibench_logo.png" width="560"/>

# Notebook 02 — Evaluating your generated designs

> **Colab users:** click **File ➜ Save a copy in Drive** before editing so your changes persist.

## Where we are in the workshop

By the end of **Notebook 01** you had a stack of generated designs sitting in a folder. They *look* like beams — sort of. The optimizer's designs look sharper. The generator's designs look blurrier. Do those two sentences tell us anything useful?

Not really. A picture tells you a design is *plausible*. A picture does not tell you:

- Whether the design **obeys the physical rules** (the constraints).
- Whether the design is **actually stiff** under the load it was designed for.
- Whether the model is producing **varied** designs or secretly copying one.
- Whether a generated design is **a better starting point** for the classical optimizer than a blank slate.

This notebook will explore metrics which answer these questions.

## What this notebook is — and isn't

This is *not* a survey of every generative-model metric in the literature. There are many (MMD, DPP, FID, coverage, precision/recall, …) and a real benchmark report would use several. We're going to stick with the four questions above because each one maps **directly** onto a single method on `problem` that you already met in Notebook 00:

| Engineering question                        | Who answers it      |
|----------------------------------------------|---------------------|
| Does the design obey the rules?              | `problem.check_constraints(...)` |
| Does the design actually work?               | `problem.simulate(...)` |
| Is the model producing varied designs?       | *(one line of NumPy)* |
| Does it help the classical optimizer?        | `problem.optimize(...)` |

That's the pedagogical point: **the benchmark's evaluation interface is the same interface we've been using the whole time.** 

## Install dependencies (Colab / fresh env only)

Skip this if your local environment already has `engibench` and `engiopt` installed.

In [ ]:
import subprocess, sys

IN_COLAB = "google.colab" in sys.modules
FORCE_INSTALL = False  # flip to True to force install locally

if IN_COLAB or FORCE_INSTALL:
    def _pip(pkgs): subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs])
    _pip(["engibench[all]", "matplotlib", "tqdm"])
    _pip(["git+https://github.com/IDEALLab/EngiOpt.git@codex/dcc26-workshop-notebooks#egg=engiopt"])
    try:
        import torch  # noqa: F401
    except Exception:
        _pip(["torch", "torchvision"])
    print("Install complete.")
else:
    print("Using current environment. Set FORCE_INSTALL=True to install here.")


In [ ]:
import json
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch as th

from engibench.utils.all_problems import BUILTIN_PROBLEMS
from engiopt.workshops.dcc26.notebook_helpers import (
    mean_pairwise_l2,
    rebuild_notebook01_artifacts,
    show_design_set,
    show_feasibility_bars,
    show_objective_comparison,
    show_optimization_trajectories,
    show_pairwise_distance_heatmap,
    show_volfrac_analysis,
)

SEED = 7
random.seed(SEED); np.random.seed(SEED); th.manual_seed(SEED)

if th.cuda.is_available():
    DEVICE = th.device("cuda")
elif th.backends.mps.is_available():
    DEVICE = th.device("mps")
else:
    DEVICE = th.device("cpu")
print("Device:", DEVICE)


---
## 0 — *Load or rebuild what Notebook 01 produced*

Notebook 01 normally saves three files into an artifacts folder:

- `generated_designs.npy` — the generator's outputs on held-out test scenarios.
- `baseline_designs.npy` — the optimizer's answers for those same scenarios.
- `conditions.json` — the scenarios themselves.

In Colab, participants sometimes open Notebook 02 in a fresh runtime or lose the `/content/dcc26_artifacts` folder. If those files are missing, this cell **rebuilds the same lightweight Notebook 01 artifact pipeline**: it loads `beams2d`, trains the same workshop CGAN-CNN for a short run, generates 24 designs, and writes the three files.

That rebuild cell can take a few minutes, especially without a GPU. It is expected; it is not a problem with the notebook.


In [ ]:
ARTIFACT_DIR = (
    Path("/content/dcc26_artifacts") if "google.colab" in sys.modules
    else Path("workshops/dcc26/artifacts")
)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

problem = BUILTIN_PROBLEMS["beams2d"](seed=SEED)

# If Notebook 01's outputs aren't in this runtime, rebuild them. The rebuild
# logic is packaged in notebook_helpers.rebuild_notebook01_artifacts so this
# cell stays short -- it is plumbing, not the lesson.
required = ["generated_designs.npy", "baseline_designs.npy", "conditions.json"]
missing = [f for f in required if not (ARTIFACT_DIR / f).exists()]
if missing:
    print(f"Missing {missing} in {ARTIFACT_DIR}.")
    print("Running the Notebook 01 artifact rebuild now so Notebook 02 can continue.")
    rebuild_notebook01_artifacts(problem, ARTIFACT_DIR, device=DEVICE, seed=SEED)
else:
    print(f"Found Notebook 01 artifacts in {ARTIFACT_DIR}.")

gen_designs = np.load(ARTIFACT_DIR / "generated_designs.npy")
baseline_designs = np.load(ARTIFACT_DIR / "baseline_designs.npy")
with open(ARTIFACT_DIR / "conditions.json") as f:
    conditions = json.load(f)

print(f"Generated designs : {gen_designs.shape}")
print(f"Baseline designs  : {baseline_designs.shape}")
print(f"Scenarios         : {len(conditions)}  (keys = {list(conditions[0].keys())})")
print(f"Artifact dir      : {ARTIFACT_DIR}")


---
## First, look at what we're evaluating

Before computing a single metric, *look at the designs.* Below are the generator's outputs for the held-out scenarios (top), each labeled with the volume fraction it was asked to hit, followed by the optimizer's answers — our **baseline** — for the very same scenarios.

Spend ten seconds comparing them by eye: the generator's beams are blurrier and grayer, the baseline's are sharper and crisper. That vague visual impression is exactly what the rest of this notebook turns into numbers you can defend.

In [ ]:
show_design_set(gen_designs, conditions, n=8,
                title="Generated designs (what the model produced)")
show_design_set(baseline_designs, conditions, n=8,
                title="Baseline / optimizer designs (same scenarios)")


---
## 1 — *Does the design obey the rules?*

This is the **feasibility** question. A generator can produce a beam that looks reasonable but quietly uses twice the material budget, or has density values outside the physical range, or breaks some solver-stability rule you didn't think to check. Low training loss gives you no protection against any of those.

In Notebook 00 we met `problem.check_constraints(design, config)`: it runs every `THEORY` and `IMPL` constraint the benchmark ships with and returns the ones that fired. Here we just call it once per generated design and count how many scenarios come back clean.

In [ ]:
def feasibility_count(designs, configs):
    feasible_flags = []
    for d, cfg in zip(designs, configs):
        # len(violations) == 0  means every constraint passed.
        violations = problem.check_constraints(design=d, config=cfg)
        feasible_flags.append(len(violations) == 0)
    return np.array(feasible_flags)


gen_feasible = feasibility_count(gen_designs, conditions)
base_feasible = feasibility_count(baseline_designs, conditions)

print(f"Generated feasible : {gen_feasible.sum()} / {len(gen_feasible)}  "
      f"({gen_feasible.mean()*100:.0f}%)")
print(f"Baseline feasible  : {base_feasible.sum()} / {len(base_feasible)}  "
      f"({base_feasible.mean()*100:.0f}%)")

In [ ]:
# Same numbers, but as a bar chart so the ratio is obvious at a glance.
show_feasibility_bars(pd.DataFrame({
    "gen_feasible": gen_feasible,
    "base_feasible": base_feasible,
}))

### Zoom in on one rule: the volume fraction

Feasibility is pass/fail across *all* the rules at once. Let's make it concrete by looking at the single most important rule on `beams2d`: the **volume fraction**.

Every scenario asks for a beam that uses a specific volume fraction — the share of the design grid allowed to be filled with material. A `volfrac` of `0.30` means *"solve this beam using material in at most 30% of the cells."* It is the budget: too much material is cheating and wastes weight; too little can't carry the load.

Because our designs are values in `[0, 1]` on a grid, the realized volume fraction of a generated design is simply its **mean pixel value**. So we can ask a very concrete question: *what percentage of generated designs actually hit the volume fraction they were asked for?* We use the benchmark's **own tolerance** for this — `beams2d` accepts a design as on-budget only if its realized fraction is within `±0.01` of the target — so this number lines up exactly with the feasibility check above.

In [ ]:
# Volume fraction = fraction of the grid filled with material = mean pixel value.
# 0.01 is the benchmark's own tolerance (engibench beams2d volume_fraction_bound),
# so this agrees with the feasibility verdict rather than an arbitrary cutoff.
volfrac_tol = 0.01

volfrac_df = pd.DataFrame({
    "target_volfrac": [c["volfrac"] for c in conditions],
    "gen_volfrac": gen_designs.reshape(len(gen_designs), -1).mean(axis=1),
    "gen_feasible": gen_feasible,
})
volfrac_df["abs_error"] = (volfrac_df["gen_volfrac"] - volfrac_df["target_volfrac"]).abs()

on_target = volfrac_df["abs_error"] <= volfrac_tol
print(f"Designs that met the volume fraction (within +/-{volfrac_tol:.2f}, the "
      f"benchmark's tolerance): {on_target.sum()} / {len(on_target)}  "
      f"({on_target.mean()*100:.0f}%)")
print(f"Mean absolute volume-fraction error: {volfrac_df['abs_error'].mean():.3f}")


In [ ]:
show_volfrac_analysis(volfrac_df, volfrac_tol=volfrac_tol)


### Which rules are being broken?

Counting feasibility tells us *how many* designs failed, not *why*. `problem.check_constraints(...)` returns a violation list — each entry has the constraint's name and a human-readable message. Tallying those names across the whole generator output set tells us which rule the model is breaking most often, which is the actionable signal: a `volume_fraction_bound` failure means *change the loss/conditioning*, while an `IMPL` solver-stability failure means *clamp the output range*.

In [ ]:
from collections import Counter

def violation_breakdown(designs, configs):
    counts = Counter()
    examples = {}  # one example message per constraint name
    cats = {}      # THEORY / IMPL / ... per name
    for d, cfg in zip(designs, configs):
        for v in problem.check_constraints(design=d, config=cfg).violations:
            name = v.constraint.check.__name__
            counts[name] += 1
            examples.setdefault(name, v.cause)
            cats.setdefault(name, str(v.constraint.categories))
    return counts, examples, cats


gen_counts, gen_examples, gen_cats = violation_breakdown(gen_designs, conditions)

if not gen_counts:
    print("No constraint violations on the generated set — nothing to break down.")
else:
    n = len(gen_designs)
    print(f"Violations on the generated set ({n} designs total):\n")
    for name, k in gen_counts.most_common():
        print(f"  {k:3d} / {n}  ({k/n*100:4.0f}%)  [{gen_cats[name]}]  {name}")
    top = gen_counts.most_common(1)[0][0]
    print(f"\nExample cause for '{top}':\n  {gen_examples[top]}")

**How to read this.** The baseline is the optimizer's output — by construction it should pass almost every constraint. If the generator's bar is noticeably shorter than the baseline's, the model is learning *something that looks like a beam* but not *something that obeys the benchmark rules*. That's a real failure mode and no amount of prettier pictures will fix it — you need to change the loss, the conditioning, or the architecture.

Notice the asymmetry: a design that's infeasible is disqualified regardless of how good its objective looks. **Feasibility gates performance.** We check it first for that reason.

---
## 2 — *Does the design actually work?*

This is the **performance** question. Among the designs that passed feasibility, how stiff are they *really* — measured by the physics simulator, not by pixel loss?

In Notebook 00 we met `problem.simulate(design, config)`. It takes seconds on `beams2d` and returns the engineering objective (compliance — lower is better). We run it on each generated design *and* on the matching baseline design under the *same* scenario, so we can compare apples to apples.

In [ ]:
rows = []
for i, (g, b, cfg) in enumerate(zip(gen_designs, baseline_designs, conditions)):
    problem.reset(seed=SEED + i)
    g_obj = float(problem.simulate(g, config=cfg)[0])
    problem.reset(seed=SEED + i)
    b_obj = float(problem.simulate(b, config=cfg)[0])
    rows.append({
        "sample": i,
        "gen_obj": g_obj,
        "base_obj": b_obj,
        "gen_minus_base": g_obj - b_obj,
        "gen_feasible": bool(gen_feasible[i]),
        "base_feasible": bool(base_feasible[i]),
    })

results = pd.DataFrame(rows)
results.head()

In [ ]:
show_objective_comparison(results)

mean_gap = results["gen_minus_base"].mean()
win_rate = float((results["gen_obj"] < results["base_obj"]).mean())
print(f"Mean objective gap (gen − base): {mean_gap:+.1f}  "
      f"(positive = generator is worse on average)")
print(f"Generator beats baseline on    : {win_rate*100:.0f}% of scenarios")

**How to read this.** For a *simple* supervised-MSE generator like ours, a positive mean gap is expected — the optimizer is a very strong baseline, and ten epochs of MSE training won't catch it. The number we'd care about in a paper is *how big* that gap is relative to the typical objective value, whether it stays stable across re-training with different seeds, and whether a more sophisticated model (GAN, diffusion, …) closes it.

---
## 3 — *Is the model producing varied designs, or one design 24 times?*

This is the **diversity** question, and it's the one that doesn't need a physics call — it's a property of the generated set itself. A generative model that collapses to a single beam topology gets a low training loss (average over the dataset looks fine) but is useless for exploration.

The crudest-but-useful measure is *mean pairwise L2 distance* between all generated designs: average how different any two outputs are. We compute the same number for the baseline set as a sanity reference — the baseline comes from an optimizer run on diverse scenarios, so it naturally spreads out.

In [ ]:
gen_div = mean_pairwise_l2(gen_designs)
base_div = mean_pairwise_l2(baseline_designs)

print(f"Mean pairwise L2 — generated : {gen_div:.2f}")
print(f"Mean pairwise L2 — baseline  : {base_div:.2f}")
print(f"Ratio (gen / base)           : {gen_div / base_div:.2f}")

In [ ]:
show_pairwise_distance_heatmap(gen_designs, baseline_designs)

**How to read it.** All three panels share one color scale. Each heatmap cell `(i, j)` is the L2 distance between design `i` and design `j`; *viridis* runs **dark purple = small distance** (near-identical) to **bright yellow = large distance** (very different), and the diagonal is always dark (a design has zero distance to itself). The left panel is the generated set, the middle is the baseline/dataset designs for the **same scenarios**, and the right overlays both off-diagonal distance distributions with their means.

---
## 4 — *Does the generator actually speed up the optimizer?*

This is the question that, if the answer is yes, justifies the whole pipeline.

Recall the argument from Notebook 01: the optimizer is slow, the generator is fast, and the dream is to amortise. But there's a gentler version of the same dream — even if the generator's designs aren't quite optimal, they might be *a much better starting point* for the classical optimizer than a blank slate. If so, then running *(generator → optimizer)* gets you the optimizer's quality in a fraction of its normal iterations.

We test this with a tiny demo: pick three scenarios, feed the generator's design into `problem.optimize(...)` as the starting point, and watch the compliance curve. We compare where the trajectory *starts* (that's what the generator gave us for free) and where it *ends* (that's where the optimizer drives it) against the baseline's compliance for the same scenario.

> **Heads-up:** `problem.optimize(...)` runs the real FEM topology optimizer — each scenario takes ~30 seconds depending on hardware.

In [ ]:
N_WARMSTART_DEMO = 3

opt_data = []
for i in range(min(N_WARMSTART_DEMO, len(gen_designs))):
    cfg = dict(conditions[i])

    # Run the optimizer starting from the GENERATED design — this is the warmstart.
    problem.reset(seed=SEED + i)
    _, history = problem.optimize(gen_designs[i], config=cfg)
    trajectory = [float(step.obj_values[0]) for step in history]

    # The compliance of the ORIGINAL baseline design — the target to hit.
    problem.reset(seed=SEED + i)
    base_obj = float(problem.simulate(baseline_designs[i], config=cfg)[0])

    opt_data.append({
        "sample_idx": i,
        "obj_trajectory": trajectory,
        "base_obj": base_obj,
    })
    print(f"Sample {i}: start = {trajectory[0]:8.1f},  end = {trajectory[-1]:8.1f},  "
          f"baseline = {base_obj:8.1f}  ({len(trajectory)} optimizer steps)")

In [ ]:
show_optimization_trajectories(opt_data)

**How to read this.** The three numbers printed at the top of each panel are the **optimality gaps** a serious benchmark would report:

- **IOG** (Initial Optimality Gap) — how far the *generated starting point* is from the baseline. Small or negative means the model already gave the optimizer something close to the answer.
- **FOG** (Final Optimality Gap) — how far the *optimizer's output from that start* is from the baseline. Close to zero means the warmstart didn't trap the optimizer in a bad local minimum.
- **COG** (Cumulative Optimality Gap) — the shaded area. Small means the optimizer converged quickly from this start.

For our MSE generator, IOG is usually awful (pictures are blurry) but FOG recovers fast — the optimizer does its job.

---
## Putting it together

In [ ]:
summary = pd.DataFrame([{
    "feasible %":              f"{gen_feasible.mean()*100:.0f}%",
    "mean obj gap (gen−base)": f"{results['gen_minus_base'].mean():+.1f}",
    "win rate vs baseline":    f"{(results['gen_obj'] < results['base_obj']).mean()*100:.0f}%",
    "diversity (L2)":          f"{gen_div:.2f}",
    "baseline diversity (L2)": f"{base_div:.2f}",
    "warmstart IOG (mean of demo)":  f"{np.mean([d['obj_trajectory'][0]  - d['base_obj'] for d in opt_data]):+.1f}",
    "warmstart COG (mean of demo)":  f"{np.mean([d['obj_trajectory'][1]  - d['base_obj'] for d in opt_data]):+.1f}",
    "warmstart FOG (mean of demo)":  f"{np.mean([d['obj_trajectory'][-1] - d['base_obj'] for d in opt_data]):+.1f}",
}]).T.rename(columns={0: "value"})
summary

---
## Outlook: the metrics we deliberately skipped

We stuck to four questions so each one mapped onto a single `problem` method. A publication-grade evaluation would add, at minimum:

- **Distributional metrics** (MMD, DPP, FID, coverage / precision-recall) — do the *two distributions* of designs match, not just pairs of them?
- **Novelty against the training set** — is the model generalizing, or quietly copy-pasting designs it memorised?
- **Spatial / material-usage comparisons** — *where* does each set place material on average?
- **Per-scenario breakdowns** — does the generator fail uniformly, or only on certain condition ranges?

---
## Reflect before moving on

1. Look at the seven-row summary. Which single number would change your mind the most if it were very good, or very bad? Why that one and not the others?
2. Suppose the feasibility rate is high but the mean objective gap is also high. What does that mean about the generator — is it being too *safe*, or too *dumb*, and how would you tell the difference?
3. The warmstart demo compared *generator-started* optimization to a baseline. What would be a fairer comparison for claiming the generator *helps* the optimizer? (Hint: Notebook 00's `problem.optimize(start, cfg)` started from a uniform field.)

## Next

You've now seen the full workflow of benchmark-driven research in engineering design: **consume** a benchmark (Notebook 00), **train against** it (Notebook 01), and **evaluate on** it (this notebook). **Notebook 03** flips the perspective and shows what it takes to *build* a new EngiBench problem of your own — answering the same eight researcher's questions from Notebook 00, but in code you write yourself.